# PySpark Learning

In [ ]:
import pyspark
print(pyspark.__version__)

## 1. Creating a Spark Session

If we're using Databricks, it automatically provides:

- spark  # SparkSession
- sc     # SparkContext
- sqlContext

In [ ]:

from pyspark.sql import SparkSession

# with PySpark locally
spark = SparkSession.builder \
        .appName("pyspark_example") \
        .getOrCreate()

print(spark)

## 2. Reading Data

### 2.1 Download data

In [ ]:
import requests
from pathlib import Path

if Path("BigMart Sales.csv").is_file():
    print("BigMart Sales.csv already exists, skipping download")
else:
    print("Downloading BigMart Sales.csv")
    request = requests.get("https://github.com/anshlambagit/PySpark-Full-Course/raw/refs/heads/main/DATA%20and%20NOTEBOOK/BigMart%20Sales.csv")
    with open("BigMart Sales.csv", "wb") as f:
        f.write(request.content)

if Path("drivers.json").is_file():
    print("drivers.json already exists, skipping download")
else:
    print("Downloading drivers.json")
    request = requests.get("https://github.com/anshlambagit/PySpark-Full-Course/raw/refs/heads/main/DATA%20and%20NOTEBOOK/drivers.json")
    with open("drivers.json", "wb") as f:
        f.write(request.content) 

### 2.2 Data Reading JSON


In [ ]:
df_json = spark.read.format('json').option('inferSchema',True) \
                    .option('header', True) \
                    .option('multiLine', False)\
                    .load('drivers.json')

df_json.show()

# In Databricks (DB)
# df_json.display()

### 2.3 Data Reading CSV


In [ ]:
df = spark.read.format('csv').option('inferSchema',True)\
                             .option('header',True)\
                             .load('BigMart Sales.csv')

# or
""" 
df = spark.read.csv(
    "BigMart Sales.csv",
    header=True,
    inferSchema=True
)
 """

df.show()

# In Databricks, we use:
# df.display()

Data Reading Utils

In [ ]:
# In DB
# dbutils.fs.ls('/FileStore/tables/')

##  3. Schema Definition


In [ ]:
df.printSchema()


### 3.1 DDL SCHEMA (Option 1)

DDL is a subset of SQL, it's the part used to define database structure.

In [ ]:
# Altering Item_Weight type
my_ddl_schema = '''
                    Item_Identifier STRING,
                    Item_Weight STRING,
                    Item_Fat_Content STRING, 
                    Item_Visibility DOUBLE,
                    Item_Type STRING,
                    Item_MRP DOUBLE,
                    Outlet_Identifier STRING,
                    Outlet_Establishment_Year INT,
                    Outlet_Size STRING,
                    Outlet_Location_Type STRING, 
                    Outlet_Type STRING,
                    Item_Outlet_Sales DOUBLE 

                ''' 

In [ ]:
df_new = spark.read.format('csv')\
            .schema(my_ddl_schema)\
            .option('header',True)\
            .load('BigMart Sales.csv') 

df_new.printSchema()



### 3.2 StructType() Schema (Option 2)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType

my_strct_schema = StructType([ 
    StructField('Item_Identifier',StringType(),True), 
    StructField('Item_Weight',StringType(),True), 
    StructField('Item_Fat_Content',StringType(),True), 
    StructField('Item_Visibility',StringType(),True), 
    StructField('Item_Type',StringType(),True), 
    StructField('Item_MRP',StringType(),True), 
    StructField('Outlet_Identifier',StringType(),True), 
    StructField('Outlet_Establishment_Year',StringType(),True), 
    StructField('Outlet_Size',StringType(),True), 
    StructField('Outlet_Location_Type',StringType(),True), 
    StructField('Outlet_Type',StringType(),True), 
    StructField('Item_Outlet_Sales',StringType(),True)
])


In [ ]:
df_new = spark.read.format('csv')\
               .schema(my_strct_schema)\
               .option('header',True)\
               .load('BigMart Sales.csv')

df_new.printSchema()


## 4. TRANSFORMATIONS


### 4.0 Select and Filter

In [ ]:
from pyspark.sql.functions import *


SELECT


In [ ]:
df.select(col('Item_Identifier'),col('Item_Weight'),col('Item_Fat_Content')).show()


ALIAS


In [ ]:
df.select(col('Item_Identifier').alias('Item_ID')).show()


FILTER


In [ ]:
# print("All Item_Fat_Content as Regular")
# df.filter(col('Item_Fat_Content')=='Regular').show()

# print("Item_type as Soft Drinks and Item_Weight < 10")
# df.filter((col('Item_Type') == 'Soft Drinks') & (col('Item_Weight')<10)).show()

print("Outlet_Size as Null and Outlet_Location_Type as Tier 1 or Tier 2")
df.filter((col('Outlet_Size').isNull()) & (col('Outlet_Location_Type').isin('Tier 1','Tier 2'))).show()


Limit


In [ ]:
df.limit(10).show()


withColumnRenamed


In [ ]:
df.withColumnRenamed('Item_Weight','Item_Wt').show()


### 4.1 withColumn


Used to create or replace a column -> `df.withColumn("column_name", expression)`

In [ ]:
df_new = df.withColumn('flag',lit("new")) # lit() creates a literal (constant) value column.
# Creates a column where every row contains "new".

df_new = df_new.withColumn('multiply',col('Item_Weight')*col('Item_MRP'))

df_new.show()


**Regular Expression (regexp or regex)** is a pattern used to match or manipulate text, like grep of Linux. Regex lets you match patterns, not just exact words.

**Used in:** Data cleaning, Feature engineering, Validation (checking formats)

In [ ]:
df_new = df.withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Regular","Reg"))\
               .withColumn('Item_Fat_Content',regexp_replace(col('Item_Fat_Content'),"Low Fat","Lf"))

# regexp_replace() replaces text using the regular expression pattern.    

df_new.show()

withColumn in **conditions**

In [ ]:
from pyspark.sql.functions import when

df_new = df.withColumn(
    "Item_Weight_Group",
    when(col("Item_Weight") < 15, "light").otherwise("heavy")
)

df_new.show()

Using select() to Add Columns -> Select is prefered adding many columns.

In [ ]:
df_new = df.select(
    "*",
    (col("Item_Weight") + 1).alias("a"),
    (col("Item_Weight") + 2).alias("b"),
    (col("Item_Weight") + 3).alias("c")
)
df_new.show()

**Type Casting**

In [ ]:
df_new = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType())) 

df_new.printSchema()

### 4.2 Sort

In [ ]:
df.sort(col('Item_Weight').desc()).show()

In [ ]:
df.sort(col('Item_Visibility').asc()).show()

In [ ]:
df.sort(['Item_Weight','Item_Visibility'], ascending = [0,0]).show()
# Item_Weight and Item_Visibility in descending order ([0,0]), Item_Weight with priority

In [ ]:
df.sort(['Item_weight','Item_Visibility'], ascending = [0,1]).show()
# Item_Weight in descending order and Item_Visibility in ascending order ([0,1]), Item_Weight with priority

### 4.3 DROP

In [ ]:
df_new = df.drop('Item_Visibility','Item_Type')

df_new.show()

Drop Duplicates

In [ ]:

df_new = df.dropDuplicates() 
# Equivalent to
# df_new.distinct()

df_new.show()

In [ ]:
df_new = df.drop_duplicates(subset=['Item_Type'])
# Drop duplicates in the Item_Type column

### 4.4 UNION and UNION BY NAME

Preparing Dataframes

In [ ]:
data1 = [('1','Pedro'),
        ('2','Teo')]
schema1 = 'id STRING, name STRING' 

df1 = spark.createDataFrame(data1,schema1)

data2 = [('3','Marina'),
        ('4','Joana')]
schema2 = 'id STRING, name STRING' 

df2 = spark.createDataFrame(data2,schema2)

df1.show()
df2.show()

Union

In [ ]:
df1.union(df2).show()

In [ ]:
data1 = [('Pedro','1',),
        ('Teo','2',)]
schema1 = 'name STRING, id STRING' 

df1 = spark.createDataFrame(data1,schema1)
df1.show()

In [ ]:
df1.union(df2).show()

""" 
+-----+------+
| name|    id|
+-----+------+
|Pedro|     1|
|  Teo|     2|
|    3|Marina|
|    4| Joana|
+-----+------+
 """

# Now it's a mess. That's why we need Union by Name


Union by Name


In [ ]:
df1.unionByName(df2).show()

### 4.5 String Functions

In [ ]:
df.select('Item_Type').show()

Initcap()

In [ ]:
df.select(initcap('Item_Type')).show()

In [ ]:
df.select(upper('Item_Type').alias('upper_Item_Type')).show()

# upper convert all text in uppercase
# lower convert all text in lowercase

### 4.6 Date Functions

current_date()

In [ ]:
df_new = df.withColumn('curr_date',current_date())

df_new.show()


Date_Add()


In [ ]:
df_new = df_new.withColumn('week_after',date_add('curr_date',7))

df_new.show()
     


Date_Sub()


In [ ]:
df_new.withColumn('week_before',date_sub('curr_date',7)).show()

In [ ]:
df_new = df_new.withColumn('week_before',date_add('curr_date',-7)) 

df_new.show()



DateDIFF


In [ ]:
df_new = df_new.withColumn('datediff',datediff('week_after','curr_date'))

df_new.show()


Date_Format()


In [ ]:
df_new = df_new.withColumn('week_before',date_format('week_before','dd-MM-yyyy'))

df_new.show()


### 4.7 Handling Nulls


Dropping Nulls


In [ ]:
df.dropna('all').show()
# drops all rows with all columns as null

In [ ]:
df.dropna('any').show()
# drops all rows with at least one null column

In [ ]:
df.dropna(subset=['Outlet_Size']).show()
# drops all rows with Outlet_Size column as null

Filling Nulls

In [ ]:
df.fillna('NotAvailable').show() # Fills nulls with the string, but only in columns with string type

In [ ]:
df.fillna('NotAvailable',subset=['Outlet_Size']).show() # Fills nulls with the string, only subset columns

### 4.8 SPLIT, Indexing, Explode and array_contains


SPLIT


In [ ]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')).show() # Uses the space to split


Indexing


In [ ]:
df.withColumn('Outlet_Type',split('Outlet_Type',' ')[1]).show() # Use index to get a value of the list

Explode

In [ ]:
df_exp = df.withColumn('Outlet_Type',split('Outlet_Type',' '))

df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).show()

# Explode "duplicates" records based on list values
# I'll have the same rows, only with the exploded column being different

array_contains

In [ ]:

df_exp.withColumn('Type1_flag',array_contains('Outlet_Type','Type1')).show()

### **4.9 GroupBY**

In [ ]:

df.groupBy('Item_Type').agg(sum('Item_MRP')).show()

In [ ]:
df.groupBy('Item_Type').agg(avg('Item_MRP')).show()

In [ ]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP').alias('Total_MRP')).show()

In [ ]:
df.groupBy('Item_Type','Outlet_Size').agg(sum('Item_MRP'),avg('Item_MRP')).show()


Collect_List


In [ ]:
data = [('user1','book1'),
        ('user1','book2'),
        ('user2','book2'),
        ('user2','book4'),
        ('user3','book1')]

schema = 'user string, book string'

df_book = spark.createDataFrame(data,schema)

df_book.show()

In [ ]:
df_book.groupBy('user').agg(collect_list('book')).show()   



PIVOT


In [ ]:


df.select('Item_Type','Outlet_Size','Item_MRP').show()

In [ ]:
df.groupBy('Item_Type').pivot('Outlet_Size').agg(avg('Item_MRP')).show()



### 4.10 When-Otherwise


In [ ]:
df_new = df.withColumn('veg_flag',when(col('Item_Type')=='Meat','Non-Veg').otherwise('Veg'))
df_new.show()

In [ ]:
df_new.withColumn('veg_exp_flag',when(((col('veg_flag')=='Veg') & (col('Item_MRP')<100)),'Veg_Inexpensive')\
                            .when((col('veg_flag')=='Veg') & (col('Item_MRP')>100),'Veg_Expensive')\
                            .otherwise('Non_Veg')).show() 

### 4.11 JOINS


In [ ]:
dataj1 = [('1','joao','d01'),
          ('2','andre','d02'),
          ('3','pedro','d03'),
          ('4','lucas','d03'),
          ('5','otavio','d05'),
          ('6','fausto','d06')] 

schemaj1 = 'emp_id STRING, emp_name STRING, dept_id STRING' 

df1 = spark.createDataFrame(dataj1,schemaj1)

dataj2 = [('d01','HR'),
          ('d02','Marketing'),
          ('d03','Accounts'),
          ('d04','IT'),
          ('d05','Finance')]

schemaj2 = 'dept_id STRING, department STRING'

df2 = spark.createDataFrame(dataj2,schemaj2)

df1.show()
df2.show()


Inner Join


In [ ]:
df1.join(df2, df1['dept_id']==df2['dept_id'],'inner').show()


Left Join


In [ ]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'left').show()

Right Join

In [ ]:
df1.join(df2,df1['dept_id']==df2['dept_id'],'right').show()

Anti Join

In [ ]:

df1.join(df2,df1['dept_id']==df2['dept_id'],'anti').show()

### 4.12 Window Functions

**A function that performs calculations across related rows while keeping each row in the output.**

In [ ]:
from pyspark.sql.window import Window

**Core Idea**

In [ ]:
data = [
    ("Alice", "Sales", 5000),
    ("Bob", "Sales", 6000),
    ("Charlie", "Sales", 4000),
    ("David", "HR", 3500),
    ("Eve", "HR", 4200)
]

dfwf = spark.createDataFrame(data, ["name", "department", "salary"])

In [ ]:
window_spec = Window.partitionBy("department").orderBy("salary")

# Partition is equivalent to grouping rows.
# Sales rows together 
# HR rows together

In [ ]:
dfwf.withColumn("rank", rank().over(window_spec)).show()


ROW_NUMBER(): Assigns a unique sequential number to each row

In [ ]:
df.withColumn('rowCol',row_number().over(Window.orderBy('Item_Identifier'))).show()


RANK(): Assigns the same rank to equal values, but it skips rankes after ties.

DENSE_RANK(): Assigns the same rank to equal values, but it does NOT skip ranks.


In [ ]:
df.withColumn('rank',rank().over(Window.orderBy(col('Item_Identifier').desc())))\
  .withColumn('denseRank',dense_rank().over(Window.orderBy(col('Item_Identifier').desc()))).show()


Cumulative Sum with **Window Frame** (Advanced)

A **Window Frame** defines: Which rows around the current row should be used in the window calculation.

Think of the hierarchy like this:

Partition → Order → Frame

1. Partition → divides the dataset into groups
2. Order → sorts rows inside each group
3. Frame → selects a subset of rows around the current row


| Value                       | Meaning            |
| --------------------------- | ------------------ |
| `Window.unboundedPreceding` | start of partition |
| `Window.unboundedFollowing` | end of partition   |
| `Window.currentRow`         | current row        |

We can have explicit window frames too → Value = -1, 0, 1, 2, N...

In [ ]:
df.withColumn('cumsum',
              sum('Item_MRP')
              .over(Window.orderBy('Item_Type')
                          .rowsBetween(Window.unboundedPreceding,Window.currentRow))).show()

# Rows: counts physical rows
# based strictly on row positions, not values.

In [ ]:
df.withColumn('cumsum',
              sum('Item_MRP')
              .over(Window.orderBy('Item_Type'))).show()

# Default frame: rangeBetween(unboundedPreceding,currentRow)
# RANGE: groups rows with equal order values
# Important: because it's RANGE, rows with the same order value may be aggregated together.

In [ ]:

df.withColumn('totalsum',
              sum('Item_MRP')
              .over(Window.orderBy('Item_Type')
                          .rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).show()

Other window functions:
- ntile(n)
- lag()
- lead()

### 4.13 User-defined functions (UDF)

In [ ]:
def my_func(x):
    return x*x 

my_udf = udf(my_func)

df.withColumn('mynewcol',my_udf('Item_MRP')).show()

## 5. Data writing (action)

| Mode        | Behavior                                  |
| ----------- | ----------------------------------------- |
| `append`    | Adds new data to existing files           |
| `overwrite` | Deletes existing data and writes new data |
| `ignore`    | Does nothing if path exists               |
| `error`     | Throws error if path exists               |


CSV

In [ ]:
df.write.format('csv')\
        .save('CSV/data.csv')

In [ ]:
df.write.format('csv')\
        .mode('append')\
        .save('CSV/data.csv')

In [ ]:
df.write.format('csv')\
        .mode('append')\
        .option('path','/FileStore/tables/CSV/data.csv')\
        .save()
     

PARQUET

In [ ]:
df.write.format('parquet')\
.mode('overwrite')\
.option('path','/FileStore/tables/CSV/data.csv')\
.save()


In [ ]:
# Parquet table
df.write.format('parquet')\
.mode('overwrite')\
.saveAsTable('my_table')


## 6. SPARK SQL



createTempView


In [ ]:
df.createTempView('my_view')

In [ ]:
df_sql = spark.sql("select * from my_view where Item_Fat_Content = 'Regular'")
     

df_sql.show()